# Reinforcement Learning with PyTorch

In [39]:
import numpy as np
import gymnasium

In [40]:
env = gymnasium.make("FrozenLake-v1", is_slippery="False")
state = env.observation_space.contains(0)
action_size = env.action_space.n
 

alpha : learning rate; gamma : discount factor; epsilon : exploration rate; episodes: number of games

In [41]:
gamma = 0.95
epsilon = 1.0
espsilon_decay = 0.995
episodes = 1000

batch_size = 64

In [42]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

In [43]:
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.fc1 = nn.Linear(state_size, 24)
        self.fc2 = nn.Linear(24, 24)
        self.fc3 = nn.Linear(24, action_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)


In [44]:
model = DQN(state_size, action_size)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_function = nn.MSELoss()

In [45]:
memory = deque(maxlen=2000)

In [46]:
def remember(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

In [47]:
def replay(batch_size):
    if len(memory) < batch_size:
        return
    
    minibatch = random.sample(memory, batch_size)
    
    for state, action, reward, next_state, done in minibatch:
        target = reward
        if not done:
            target += gamma * torch.max(model(torch.FloatTensor(next_state)))

        output = model(torch.FloatTensor(state))[action]

        loss = loss_function(output.view(-1), torch.tensor([target], dtype=torch.float32))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
        # target_f = model(torch.FloatTensor(state))
        # target_f[action] = target
        # optimizer.zero_grad()
        # loss = loss_function(model(torch.FloatTensor(state)), target_f)
        # loss.backward()
        # optimizer.step()

In [ ]:
for episode in range(episodes):
    state_size = env.reset()[0]
    done = False
    total_reward = 0

    while not done:
        if np.random.rand() < epsilon:
            action = random.randrange(action_size) # Explore
        else:
            action = torch.argmax(model(torch.FloatTensor(state))).item() # Exploit
            remember(state, action, reward, next_state, done)
            state= next_state
            total_reward += reward
        
    replay()
    epsilon = max(epsilon * espsilon_decay, epsilon_min)
    print(f"Episode: {episode+1}/{episodes}, Total Reward: {total_reward}, Epsilon: {epsilon:.2f}")

print("Training completed.") 
